# Reproduce Results
This notebook loads the saved models and reports ROC AUC, Precision, and Recall on the train, validation, and test sets.

In [7]:
import os
import numpy as np
import pandas as pd
import sklearn.model_selection
from sklearn.metrics import roc_auc_score, precision_score, recall_score
import joblib
from utils import extract_simple_features, extract_improved_features, extract_lluc_features, extract_miki_features, extract_jose_features, extract_final_features

## Data Loading and Splitting

In [8]:
quora_df = pd.read_csv("./quora_data.csv")
A_df, test_df = sklearn.model_selection.train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = sklearn.model_selection.train_test_split(A_df, test_size=0.05, random_state=123)

## Load TF-IDF
Load the fitted vectorizers to correctly transform test features.

In [9]:
import joblib

tfidf_word = joblib.load('models/tfidf_word.pkl')
tfidf_char = joblib.load('models/tfidf_char.pkl')

## Feature Extraction

In [10]:
y_train = train_df['is_duplicate'].values
y_val = val_df['is_duplicate'].values
y_test = test_df['is_duplicate'].values

X_train_simp = extract_simple_features(train_df)
X_val_simp = extract_simple_features(val_df)
X_test_simp = extract_simple_features(test_df)

X_train_imp = extract_improved_features(train_df)
X_val_imp = extract_improved_features(val_df)
X_test_imp = extract_improved_features(test_df)

X_train_lluc_full = np.hstack((X_train_simp, extract_lluc_features(train_df)))
X_val_lluc_full = np.hstack((X_val_simp, extract_lluc_features(val_df)))
X_test_lluc_full = np.hstack((X_test_simp, extract_lluc_features(test_df)))

X_train_miki_full = np.hstack((X_train_simp, extract_miki_features(train_df, tfidf_word, tfidf_char)))
X_val_miki_full = np.hstack((X_val_simp, extract_miki_features(val_df, tfidf_word, tfidf_char)))
X_test_miki_full = np.hstack((X_test_simp, extract_miki_features(test_df, tfidf_word, tfidf_char)))

X_train_jose_full = np.hstack((X_train_simp, extract_jose_features(train_df)))
X_val_jose_full = np.hstack((X_val_simp, extract_jose_features(val_df)))
X_test_jose_full = np.hstack((X_test_simp, extract_jose_features(test_df)))

X_train_final = extract_final_features(train_df, tfidf_word, tfidf_char)
X_val_final = extract_final_features(val_df, tfidf_word, tfidf_char)
X_test_final = extract_final_features(test_df, tfidf_word, tfidf_char)

## Evaluation Function

In [11]:
def evaluate_model(model, X, y):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]
    return {
        'ROC_AUC': roc_auc_score(y, probs),
        'Precision': precision_score(y, preds),
        'Recall': recall_score(y, preds)
    }

## Load Models & Evaluate

In [12]:
model_simple = joblib.load('models/model_simple.pkl')
model_improved = joblib.load('models/model_improved.pkl')
model_miki = joblib.load('models/model_miki.pkl')
model_jose = joblib.load('models/model_jose.pkl')
model_lluc = joblib.load('models/model_lluc.pkl')
model_final = joblib.load('models/model_final.pkl')

results = {
    'Simple_Train': evaluate_model(model_simple, X_train_simp, y_train),
    'Simple_Val': evaluate_model(model_simple, X_val_simp, y_val),
    'Simple_Test': evaluate_model(model_simple, X_test_simp, y_test),
    'Improved_Train': evaluate_model(model_improved, X_train_imp, y_train),
    'Improved_Val': evaluate_model(model_improved, X_val_imp, y_val),
    'Improved_Test': evaluate_model(model_improved, X_test_imp, y_test),
    'Miki_Train': evaluate_model(model_miki, X_train_miki_full, y_train),
    'Miki_Val': evaluate_model(model_miki, X_val_miki_full, y_val),
    'Miki_Test': evaluate_model(model_miki, X_test_miki_full, y_test),
    'Jose_Train': evaluate_model(model_jose, X_train_jose_full, y_train),
    'Jose_Val': evaluate_model(model_jose, X_val_jose_full, y_val),
    'Jose_Test': evaluate_model(model_jose, X_test_jose_full, y_test),
    'Lluc_Train': evaluate_model(model_lluc, X_train_lluc_full, y_train),
    'Lluc_Val': evaluate_model(model_lluc, X_val_lluc_full, y_val),
    'Lluc_Test': evaluate_model(model_lluc, X_test_lluc_full, y_test),
    'Final_Train': evaluate_model(model_final, X_train_final, y_train),
    'Final_Val': evaluate_model(model_final, X_val_final, y_val),
    'Final_Test': evaluate_model(model_final, X_test_final, y_test)
}

results_df = pd.DataFrame(results).T
display(results_df)

,ROC_AUC,Precision,Recall
Simple_Train,0.712409,0.530087,0.382662
Simple_Val,0.710943,0.531788,0.376390
Simple_Test,0.709384,0.535848,0.380722
Improved_Train,0.727625,0.533806,0.468211
Improved_Val,0.724784,0.528325,0.454209
Improved_Test,0.725574,0.534178,0.463126
Miki_Train,0.763881,0.579075,0.468248
Miki_Val,0.760769,0.575336,0.460208
Miki_Test,0.757886,0.578958,0.466289
Jose_Train,0.769228,0.579792,0.533564
